In [6]:
import numpy as np
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.contingency_tables import Table2x2

# Example table:
#
#                    Depression
#                 Yes        No
#
# Counseling Yes   65       109
# Counseling No   243      1348
#
# These numbers roughly mimic:
# 37.4% vs 15.3%

table = np.array([
    [65, 109],
    [243, 1348]
])

# ---------------------------------------------------
# Basic rates
# ---------------------------------------------------

yes_rate = table[0, 0] / table[0].sum()
no_rate  = table[1, 0] / table[1].sum()

print("Depression rate given counseling:")
print(f"  {yes_rate:.3%}")

print("Depression rate without counseling:")
print(f"  {no_rate:.3%}")

# # ---------------------------------------------------
# # Chi-square test of independence
# # ---------------------------------------------------

# chi2, p_value, dof, expected = chi2_contingency(table)

# print("\nChi-square test")
# print("----------------")
# print(f"chi2 statistic : {chi2:.4f}")
# print(f"p-value        : {p_value:.6g}")
# print(f"degrees freedom: {dof}")

# print("\nExpected counts under independence:")
# print(expected)

# # ---------------------------------------------------
# # Fisher exact test
# # ---------------------------------------------------

# odds_ratio_fisher, fisher_p = fisher_exact(table)

# print("\nFisher exact test")
# print("------------------")
# print(f"odds ratio: {odds_ratio_fisher:.4f}")
# print(f"p-value   : {fisher_p:.6g}")

# ---------------------------------------------------
# Odds ratio + confidence interval
# ---------------------------------------------------

t = Table2x2(table)

print("\nOdds ratio analysis")
print("--------------------")
print(f"odds ratio      : {t.oddsratio:.4f}")

ci_low, ci_high = t.oddsratio_confint()

print(f"95% CI          : ({ci_low:.4f}, {ci_high:.4f})")

# ---------------------------------------------------
# Interpretation
# ---------------------------------------------------

if ci_low > 1:
    print("\nAssociation appears positive.")
else:
    print("\nNo clear positive association.")

Depression rate given counseling:
  37.356%
Depression rate without counseling:
  15.273%

Odds ratio analysis
--------------------
odds ratio      : 3.3080
95% CI          : (2.3636, 4.6298)

Association appears positive.


In [1]:
import importlib
import numpy as np

from src import analysis
from src.noise import gaussian
from src.release import noisy_float
from statsmodels.stats.contingency_tables import Table2x2

analysis = importlib.reload(analysis)

rng = np.random.default_rng(seed=42)
noise_factory = gaussian(loc=0, scale=1)
count_11 = noisy_float(65, noise_factory, seed=rng)
count_10 = noisy_float(109, noise_factory, seed=rng)
count_01 = noisy_float(243, noise_factory, seed=rng)
count_00 = noisy_float(1348, noise_factory, seed=rng)

noisy_table = analysis.NoisyTable2x2.from_cells(count_11, count_10, count_01, count_00)

print("Depression rate given counseling:")
print(f"  {float(noisy_table.table[0, 0] / noisy_table.table[0].sum()):.3%}")
print("Depression rate without counseling:")
print(f"  {float(noisy_table.table[1, 0] / noisy_table.table[1].sum()):.3%}")

# ---------------------------------------------------
# Classical CI on a fixed table
# ---------------------------------------------------

true_table = np.array([[65.0, 109.0], [243.0, 1348.0]], dtype=float)
t = Table2x2(true_table)
plain_low, plain_high = t.oddsratio_confint()

print("\nClassical statsmodels CI on the fixed table:")
print(f"  ({plain_low:.6f}, {plain_high:.6f})")

# ---------------------------------------------------
# DP-aware quantile CIs on NoisyTable2x2
# ---------------------------------------------------

quant_low_no_sampling, quant_high_no_sampling = noisy_table.oddsratio_confint_noisy_quantile(
    n=5000,
    seed=123,
    correction=0.0,
    alpha=0.05,
    include_sampling=False,
)

quant_low_sampling, quant_high_sampling = noisy_table.oddsratio_confint_noisy_quantile(
    n=5000,
    seed=123,
    correction=0.0,
    alpha=0.05,
    include_sampling=True,
)

print("\nDP quantile CI (posterior only; no sampling layer):")
print(f"  ({quant_low_no_sampling:.6f}, {quant_high_no_sampling:.6f})")
print(f"  width = {quant_high_no_sampling - quant_low_no_sampling:.6e}")

print("\nDP quantile CI (posterior + sampling uncertainty):")
print(f"  ({quant_low_sampling:.6f}, {quant_high_sampling:.6f})")
print(f"  width = {quant_high_sampling - quant_low_sampling:.6e}")

if quant_low_sampling > 1:
    print("\nAssociation appears positive under combined uncertainty.")
else:
    print("\nNo clear positive association under combined uncertainty.")

Depression rate given counseling:
  37.691%
Depression rate without counseling:
  15.304%

Classical statsmodels CI on the fixed table:
  (2.363633, 4.629785)

DP quantile CI (posterior only; no sampling layer):
  (3.226627, 3.470483)
  width = 2.438560e-01

DP quantile CI (posterior + sampling uncertainty):
  (2.379441, 4.671005)
  width = 2.291564e+00

Association appears positive under combined uncertainty.


## Interpreting the DP-aware quantile CIs

- `NoisyTable2x2(...).oddsratio_confint_noisy_quantile(..., include_sampling=False)` returns a posterior OR quantile CI without re-sampling study participants.
- `NoisyTable2x2(...).oddsratio_confint_noisy_quantile(..., include_sampling=True)` returns a posterior OR quantile CI with sampling uncertainty included.
- Use `include_sampling=True` when you want the final interval to reflect both release noise and the study sampling layer.